In [ ]:
import torch
import yaml

from train import SSMTranslatorTrainer 
from ssm import SSMTranslator, SSMTranslatorConfig

def does_run(model, batch_size, input_length, target_length, device):
    input_ids = torch.ones((batch_size, input_length), dtype=torch.long).to(device)
    target_ids = torch.ones((batch_size, target_length), dtype=torch.long).to(device)

    try:
        loss = model(inp=input_ids, target=target_ids, decode_method="forced")
        loss.backward()
        return True
    except Exception as e:
        print(f"Model failed to run: {e}")
        return False
    
def binary_search_max_input_length(model, batch_size, device, min_length=1, max_length=2048):
    model.train()
    while min_length < max_length:
        print(f"Testing input length: {min_length} to {max_length}")
        mid_length = (min_length + max_length) // 2
        if does_run(model, batch_size, mid_length // 2, (mid_length - 1) // 2 + 1, device):
            min_length = mid_length + 1
        else:
            max_length = mid_length
    return min_length - 1

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

model_config = SSMTranslatorConfig(**yaml.safe_load(open("../configs/ssm_model_config_56m.yaml", "r")))
model = SSMTranslatorTrainer(SSMTranslator(model_config)).to(device)

batch_size = 4
max_input_length = binary_search_max_input_length(model, batch_size, device)
print(f"Maximum input length for batch size {batch_size}: {max_input_length}")

# 256 x 4 seems safe for 16 gb. We'll go 256 x 8 for 32 gb


Testing input length: 1 to 2048
Model failed to run: MPS backend out of memory (MPS allocated: 16.75 GiB, other allocations: 6.91 MiB, max allowed: 18.13 GiB). Tried to allocate 2.99 GiB on shared pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).
Testing input length: 1 to 1024
Model failed to run: MPS backend out of memory (MPS allocated: 17.58 GiB, other allocations: 3.00 MiB, max allowed: 18.13 GiB). Tried to allocate 573.75 MiB on shared pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).
Testing input length: 1 to 512
Testing input length: 257 to 512


KeyboardInterrupt: 

In [ ]:
# get length distribution of training data

import pandas as pd
import sentencepiece as sp

proc_en = sp.SentencePieceProcessor()
proc_en.Load("../vocab/en.model")
proc_fr = sp.SentencePieceProcessor()
proc_fr.Load("../vocab/fr.model")

df = pd.read_csv("../data/train.csv", nrows=10000)  # already shuffled
df["en"] = df["en"].apply(lambda x: len(proc_en.Encode(x, add_bos=True, add_eos=True)))
df["fr"] = df["fr"].apply(lambda x: len(proc_fr.Encode(x, add_bos=True, add_eos=True)))

print(df.describe())

                 en            fr
count  10000.000000  10000.000000
mean      34.851100     42.716000
std       46.145648     50.324046
min        3.000000      3.000000
25%       19.000000     24.000000
50%       28.000000     35.000000
75%       41.000000     51.000000
max     3666.000000   3712.000000


In [2]:
# how many do we miss with 256? (pct)

print((df["en"] > 256 / 2).mean())
print((df["fr"] > 256 / 2).mean())

0.0122
0.0202
